In [8]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('smart_meter_data.csv')

In [3]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Timestamp             5000 non-null   object 
 1   Electricity_Consumed  5000 non-null   float64
 2   Temperature           5000 non-null   float64
 3   Humidity              5000 non-null   float64
 4   Wind_Speed            5000 non-null   float64
 5   Avg_Past_Consumption  5000 non-null   float64
 6   Anomaly_Label         5000 non-null   object 
dtypes: float64(5), object(2)
memory usage: 273.6+ KB
None


In [22]:
df['HouseName'] = ['House' + str(i+1) for i in range(len(df))]
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Timestamp             5000 non-null   datetime64[ns]
 1   Electricity_Consumed  5000 non-null   float64       
 2   Temperature           5000 non-null   float64       
 3   Humidity              5000 non-null   float64       
 4   Wind_Speed            5000 non-null   float64       
 5   Avg_Past_Consumption  5000 non-null   float64       
 6   Anomaly_Label         5000 non-null   object        
 7   HouseName             5000 non-null   object        
 8   MA_24                 5000 non-null   float64       
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 351.7+ KB
None


In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
print(df['Timestamp'].dtype)

datetime64[ns]


In [16]:
df['Electricity_Consumed'] = df['Electricity_Consumed'] * 100

In [17]:
house = 'House1'
house_df = df[df['HouseName'] == house]


In [25]:
import plotly.express as px

avg_df = (
    df.groupby('HouseName')['Electricity_Consumed']
      .mean()
      .reset_index()
)


In [28]:
fig = px.line(
    avg_df,
    x='HouseName',
    y='Electricity_Consumed',
    markers=True,
    title='Average Electricity Consumption per House'
)

fig.update_layout(
    xaxis_title='House',
    yaxis_title='Average Electricity Consumed (kWh)',
    template='plotly_dark'
)

fig.show()


In [29]:
df = df.sort_values(by=['HouseName', 'Timestamp'])

In [30]:
df['MA_24'] = (
    df.groupby('HouseName')['Electricity_Consumed']
      .transform(lambda x: x.rolling(window=24, min_periods=1).mean())
)


In [32]:
house_df = df[df['HouseName'] == 'House1']

print(house_df.shape)
print(house_df['Timestamp'].nunique())
print(house_df[['Timestamp', 'Electricity_Consumed']].head(10))


(1, 9)
1
   Timestamp  Electricity_Consumed
0 2024-01-01          4.577857e+07


In [33]:
rows_per_house = 500

df['HouseName'] = (
    df.index // rows_per_house + 1
).astype(str)

df['HouseName'] = 'House' + df['HouseName']


In [34]:
import plotly.express as px

house_df = df[df['HouseName'] == 'House1']

fig = px.line(
    house_df,
    x='Timestamp',
    y=['Electricity_Consumed', 'MA_24'],
    title='Electricity Consumption vs Moving Average (House1)'
)

fig.update_layout(
    xaxis_title='Time',
    yaxis_title='Electricity Consumed (kWh)',
    template='plotly_dark'
)

fig.show()
